## Checking for Hardware configuration


In [1]:
!uname -a && cat /etc/*release

Linux 55a4fd97a689 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [2]:
!pwd
!ls -la

/content
total 24
drwxr-xr-x 1 root root 4096 May  7 16:15 .
drwxr-xr-x 1 root root 4096 May  7 15:56 ..
drwxr-xr-x 4 root root 4096 Apr 16 13:33 .config
drwxr-xr-x 1 root root 4096 Apr 16 13:33 sample_data
-rw-r--r-- 1 root root 1601 May  7 16:15 VectorAddCuda.cu
-rw-r--r-- 1 root root 1745 May  7 16:15 VectorAddGrid.cu


In [3]:
!nvidia-smi

Thu May  7 16:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
!nvcc -arch=sm_75 hello.cu -o hello
!./hello

Hello, World from GPU!


## Testing with N = 10000000

In [24]:
!nvcc -arch=sm_75 VectorAddCuda.cu -o VectorAddCuda
!./VectorAddCuda

out[0] = 3.000000
PASSED


In [25]:
! nvprof ./VectorAddCuda

==23489== NVPROF is profiling process 23489, command: ./VectorAddCuda
out[0] = 3.000000
PASSED
==23489== Profiling application: ./VectorAddCuda
==23489== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   93.45%  606.62ms         1  606.62ms  606.62ms  606.62ms  vector_add(float*, float*, float*, int)
                    4.00%  25.959ms         1  25.959ms  25.959ms  25.959ms  [CUDA memcpy DtoH]
                    2.55%  16.554ms         2  8.2771ms  8.2662ms  8.2879ms  [CUDA memcpy HtoD]
      API calls:   77.84%  650.43ms         3  216.81ms  8.4811ms  633.47ms  cudaMemcpy
                   21.56%  180.16ms         3  60.052ms  77.713us  179.93ms  cudaMalloc
                    0.31%  2.6114ms       114  22.906us      88ns  1.4834ms  cuDeviceGetAttribute
                    0.27%  2.2228ms         3  740.95us  235.98us  998.64us  cudaFree
                    0.02%  148.30us         1  148.30us  148.30us  148.30us  

## Testing with N=10000000 and parallelized 1 block and 256 threads

In [31]:
!nvcc -arch=sm_75 VectorAddCuda.cu -o VectorAddCuda
!./VectorAddCuda

out[0] = 3.000000
PASSED


In [32]:
! nvprof ./VectorAddCuda

==27989== NVPROF is profiling process 27989, command: ./VectorAddCuda
out[0] = 3.000000
PASSED
==27989== Profiling application: ./VectorAddCuda
==27989== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   40.67%  25.896ms         1  25.896ms  25.896ms  25.896ms  [CUDA memcpy DtoH]
                   33.16%  21.112ms         1  21.112ms  21.112ms  21.112ms  vector_add(float*, float*, float*, int)
                   26.17%  16.660ms         2  8.3298ms  8.2180ms  8.4416ms  [CUDA memcpy HtoD]
      API calls:   70.78%  167.06ms         3  55.686ms  71.572us  166.90ms  cudaMalloc
                   27.50%  64.897ms         3  21.632ms  8.4476ms  47.803ms  cudaMemcpy
                    1.00%  2.3583ms         3  786.12us  202.73us  1.1058ms  cudaFree
                    0.65%  1.5356ms       114  13.470us     114ns  782.87us  cuDeviceGetAttribute
                    0.06%  140.04us         1  140.04us  140.04us  140.04us  

## Testing with N/256 blocks and 256 threads

In [35]:
!nvcc -arch=sm_75 VectorAddGrid.cu -o VectorAddGrid
!./VectorAddGrid

out[0] = 3.000000
PASSED


In [36]:
! nvprof ./VectorAddGrid

==31062== NVPROF is profiling process 31062, command: ./VectorAddGrid
out[0] = 3.000000
PASSED
==31062== Profiling application: ./VectorAddGrid
==31062== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   59.20%  25.480ms         1  25.480ms  25.480ms  25.480ms  [CUDA memcpy DtoH]
                   39.76%  17.112ms         2  8.5558ms  8.4481ms  8.6635ms  [CUDA memcpy HtoD]
                    1.05%  450.07us         1  450.07us  450.07us  450.07us  vector_add(float*, float*, float*, int)
      API calls:   76.27%  154.92ms         3  51.641ms  71.819us  154.76ms  cudaMalloc
                   21.77%  44.230ms         3  14.743ms  8.6788ms  26.698ms  cudaMemcpy
                    1.17%  2.3852ms         3  795.07us  222.96us  1.0969ms  cudaFree
                    0.70%  1.4199ms       114  12.455us      86ns  790.32us  cuDeviceGetAttribute
                    0.07%  144.25us         1  144.25us  144.25us  144.25us  

- The CUDA parallel kernel became much faster, but the overall runtime is limited by memory transfer overhead.
- So the kernel speedup is approximately:606 ms / 0.450 ms ≈ 1346x faster
-


## Testing with 1,000,000,000 without parallization

In [17]:
!nvcc -arch=sm_75 VectorAddCuda.cu -o VectorAddCuda
!./VectorAddCuda

^C


In [18]:
! nvprof ./VectorAddCuda

==20334== NVPROF is profiling process 20334, command: ./VectorAddCuda
==20334== Profiling application: ./VectorAddCuda
==20334== Profiling result:
No kernels were profiled.
No API activities were profiled.
==20334== Warning: Some profiling data are not recorded.
======== Error: Application received signal 9


## Testing with 1,000,000,000 with 256 thread on single block

In [4]:
!nvcc -arch=sm_75 VectorAddCuda.cu -o VectorAddCuda
! nvprof ./VectorAddCuda


==5755== NVPROF is profiling process 5755, command: ./VectorAddCuda
==5755== Profiling application: ./VectorAddCuda
==5755== Profiling result:
No kernels were profiled.
No API activities were profiled.
==5755== Warning: Some profiling data are not recorded.
======== Error: Application received signal 9


## Testing with 1,000,000,000 with 256 thread on 256/N grid

In [5]:
!nvcc -arch=sm_75 VectorAddGrid.cu -o VectorAddGrid


In [6]:
! nvprof ./VectorAddGrid

==6542== NVPROF is profiling process 6542, command: ./VectorAddGrid
==6542== Profiling application: ./VectorAddGrid
==6542== Profiling result:
No kernels were profiled.
No API activities were profiled.
==6542== Warning: Some profiling data are not recorded.
======== Error: Application received signal 9
